# 15 · Results, hypotheses and figures

The only route from experiments to the write-up. It reads every logged metric, builds the results tables, decides each pilot hypothesis with the rule fixed in advance in the pilot note, and draws the figures. No number is typed by hand: if a run changes, rerun this notebook and every table changes with it. The decision rules live in `cffm/report.py`, where they are unit-tested.

| | |
|---|---|
| **Kaggle settings** | Accelerator **GPU T4 x2**, Internet **on** |
| **Inputs to attach** | `cffm-net-pilot` and the **saved outputs of notebooks 04, 05, 07, 08 and 09** |
| **Expected runtime** | a few minutes |
| **Produces** | `results/` with tables (Markdown, LaTeX, CSV) and figures (PNG, PDF) |

In [ ]:
# --------------------------------------------------------------------------------------------
# Setup: the same cell opens every notebook. It finds the code, installs it, and checks the GPU.
#
# On Kaggle the uploaded cffm-net-pilot.zip appears, already unzipped, somewhere under
# /kaggle/input. Inputs are read-only, so we copy the project to /kaggle/working and install
# it from there ("pip install -e" makes `import cffm` work in this kernel, in DataLoader
# workers, and in the extra processes Ultralytics starts when it trains on two GPUs).
# Locally, notebooks live in cffm-net-pilot/notebooks/<phase>/; the project is the first folder up
# that contains src/cffm.
# --------------------------------------------------------------------------------------------
import glob, shutil, subprocess, sys
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").exists()
if ON_KAGGLE:
    # up to five levels deep: /kaggle/input/<slug>/, /kaggle/input/<slug>/cffm-net-pilot/,
    # /kaggle/input/datasets/<owner>/<slug>/..., whichever layout this Kaggle version uses
    hits = [Path(p).parent for k in range(1, 6) for p in glob.glob("/kaggle/input" + "/*" * k + "/pyproject.toml")
            if (Path(p).parent / "src" / "cffm").is_dir()]
    assert hits, "Add the 'cffm-net-pilot' dataset (the uploaded zip) as an input to this notebook."
    PROJECT = Path("/kaggle/working/cffm-net-pilot")
    if not PROJECT.exists():
        shutil.copytree(hits[0], PROJECT, copy_function=shutil.copyfile)
        for p in [PROJECT, *PROJECT.rglob("*")]:     # the input was read-only; our copy must be writable
            p.chmod(p.stat().st_mode | 0o200)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.171",
                    "faster-coco-eval>=1.6.7", "cloudpickle"], check=True)   # cloudpickle: two-GPU launcher
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(PROJECT)], check=True)
else:
    PROJECT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "cffm").is_dir())
sys.path.insert(0, str(PROJECT / "src"))

import cffm
from cffm import env as cenv, pipeline
print("cffm", cffm.__version__, "imported from", Path(cffm.__file__).parent)   # must be inside PROJECT
env = cenv.setup()                       # prints platform, GPUs and paths; moves into the project folder
plan = pipeline.load_plan(env.project)   # configs/pilot.yaml: every run and its settings

In [ ]:
import json
from cffm import report
ev = pipeline.find_metrics(env, "eval")       # full-validation-set metrics of every trained run
pr = pipeline.find_metrics(env, "probe")      # metrics under each test-time degradation
lat = {}
for root in [env.runs, *[r for r in pipeline.INPUT_ROOTS if r.exists()]]:
    for f in root.glob("**/latency/latency.json"):
        lat.update({k: v for k, v in json.loads(f.read_text()).items() if k not in lat})
print(len(ev), "evaluated runs,", len(pr), "probe results,", len(lat), "latency entries")
out = env.project / "results"; out.mkdir(exist_ok=True)

## 1. Main results

AP is COCO AP50-95 on the full validation set. AP_vt / AP_t / AP_s are the AI-TOD bands (<8, 8-16, 16-32 px); AP_small is COCO's <32 px. Cost is measured on one T4, FP16, batch 1.

In [ ]:
main = report.main_table(ev, lat)
main.round(4)

## 2. The hypotheses, decided by the rules written down before training

From the pilot note (Table 4). Thresholds are in AP points (0.015 = 1.5 points). A difference inside the
threshold is **inconclusive**, not a win: with one seed per configuration, run-to-run variation is about 1 to
1.5 points. *Against* means the difference is as large as the threshold in the wrong direction.

In [ ]:
hyp = report.decide(ev, pr, lat, plan["probe"]["kinds"])
hyp

## 3. Figures

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({"font.family": "serif", "axes.spines.top": False, "axes.spines.right": False})

llvip = main[main.index.str.startswith("llvip")]
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.barh(llvip.index, llvip["AP"], color=["#3B8B66" if "cffm" in i else "#66707B" for i in llvip.index])
ax.set_xlabel("COCO AP50-95, LLVIP (full validation set)"); ax.invert_yaxis(); ax.grid(axis="x", alpha=0.3)
plt.tight_layout(); plt.savefig(out / "fig_llvip_ap.png", dpi=200); plt.savefig(out / "fig_llvip_ap.pdf"); plt.show()

m3 = main.loc[[r for r in ("m3fd_concat", "m3fd_cffm") if r in main.index], ["AP_vt", "AP_t", "AP_s", "AP"]]
if len(m3):
    ax = m3.T.plot(kind="bar", figsize=(7, 3.6), color=["#66707B", "#3B8B66"], rot=0)
    ax.set_ylabel("AP50-95"); ax.set_title("M3FD by object size"); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout(); plt.savefig(out / "fig_m3fd_sizes.png", dpi=200); plt.savefig(out / "fig_m3fd_sizes.pdf"); plt.show()

In [ ]:
probe = report.probe_table(pr, plan["probe"]["runs"], plan["probe"]["kinds"], plan["probe"]["flagged"])
if len(probe):
    ax = probe.plot(kind="barh", figsize=(9, 5.5), width=0.8)
    ax.set_xlabel("COCO AP50-95 under test-time degradation (LLVIP)"); ax.invert_yaxis(); ax.grid(axis="x", alpha=0.3)
    plt.tight_layout(); plt.savefig(out / "fig_probe.png", dpi=200); plt.savefig(out / "fig_probe.pdf"); plt.show()
probe.round(4)

## 4. Tables for the write-up

Markdown for notes, CSV for spreadsheets, and LaTeX in the dossier's rule-free style for the paper.

In [ ]:
main.round(4).to_csv(out / "main_results.csv")
hyp.to_csv(out / "hypotheses.csv")
if len(probe): probe.round(4).to_csv(out / "probe.csv")
(out / "tables.md").write_text("## Main results\n\n" + report.to_markdown(main) +
                               "\n## Hypotheses\n\n" + report.to_markdown(hyp) +
                               ("\n## Degradation probe\n\n" + report.to_markdown(probe) if len(probe) else ""))
(out / "tables.tex").write_text(report.to_latex(main, "Pilot results, one seed per configuration.", "tab:pilot")
                                + "\n" + report.to_latex(hyp, "Pilot hypotheses and verdicts.", "tab:pilot-hyp"))
print(sorted(p.name for p in out.iterdir()))

### Keep the results

On Kaggle, click **Save Version → Save & Run All (Commit)**. The notebook then reruns top to bottom in the
background, and its `runs/` folder becomes an output that notebooks **09** and **15** can attach as an input
(*Add Input → Your Work → this notebook*). Without a saved version, the checkpoints vanish when the session ends.